<a href="https://colab.research.google.com/github/Lateephah/Applied-Search-Intelligence-System/blob/main/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*


###My Response:

I chose the Refresh / Content Opportunity Scoring lane because it focuses on helping content teams decide which pages should be reviewed first instead of updating every page indiscriminately. After running run_all.py on the content_refresh_anonymized dataset, I learned how the baseline score and decision tree were used to prioritize pages using Precision@50. That made me curious about whether observable search and content signals can identify worthwhile review candidates. This matters because recommending that a client "refresh everything" is costly and impractical; I want to investigate whether the available data can support more targeted, evidence-based review decisions.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

###Response:

**Decision it improves:**

 Whether a content team should prioritize refreshing old pages vs. writing new ones.

**Who acts on it:**

 Content/SEO strategists deciding where to spend limited editing time.

**Cost of a wrong call:**

 If we say "freshness helps" and it doesn't, teams waste hours refreshing pages for no ranking gain. If we say "freshness doesn't matter" and it actually does, stale content keeps losing visibility and nobody notices why.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [11]:
import pandas as pd
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Lateephah/Applied-Search-Intelligence-System"
REPO_DIR = "Applied-Search-Intelligence-System"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")




In [12]:
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null

In [14]:
#This shows all the freshness tiers exist and their corresponding number of pages in each
print(df['freshness_tier'].value_counts())

freshness_tier
0-30      20480
91-180     9171
31-90       175
181+        174
Name: count, dtype: int64


Most pages (20,480 out of 30,000) were updated within the last 30 days, while relatively few pages fall into the older freshness categories. This indicates that the dataset contains pages with varying levels of freshness, allowing me to investigate whether update recency is associated with different search performance patterns.

In [15]:
# While this shows the average position by freshness tier
print(df.groupby('freshness_tier')['avg_position'].mean().sort_values())

freshness_tier
181+      11.325862
0-30      15.685166
31-90     16.538286
91-180    17.901461
Name: avg_position, dtype: float64


Of 30,000 pages, pages in the 181+ days freshness tier have the lowest average search position (11.33), while pages updated 91–180 days ago have the highest average position (17.90). This suggests there may be an association between content freshness and search performance, although this observation alone does not establish a causal relationship.

(Remember: *in Google Search Console, a lower average position means a better ranking.*)

In [16]:
# And this shows how freshness relate to the different trend direction, the "declining", "stable" and "improving"
print(pd.crosstab(df['freshness_tier'], df['trend_direction'], normalize='index').round(2))

trend_direction  down  flat   new  stable    up
freshness_tier                                 
0-30             0.51  0.04  0.10    0.19  0.15
181+             0.47  0.09  0.14    0.14  0.16
31-90            0.59  0.01  0.04    0.15  0.22
91-180           0.61  0.03  0.01    0.23  0.13


The proportion of declining pages varies across freshness tiers. For example, 61% of pages updated 91–180 days ago are trending downward compared with 51% of pages updated within the last 30 days. This suggests freshness may be one of several signals worth investigating when prioritizing pages for review.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

###My Response:

This project will provide decision support by identifying observable patterns between content freshness, search performance and page trends; and the results will be used to help prioritize which pages may be worth reviewing first based on measured data.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.